In [1]:
import random
import string
from datetime import datetime
from time import time

import numpy as np
import pandas as pd


import psycopg2
from psycopg2 import sql
from psycopg2.extensions import ISOLATION_LEVEL_AUTOCOMMIT

from pymilvus import (
    connections,
    utility,
    FieldSchema, CollectionSchema, DataType,
    Collection,
)

import torch
from transformers import BertModel, BertTokenizer

/home/nthom/miniconda3/envs/SEEK/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Create a table in a database

In [2]:
# Connect to your postgres DB
conn = psycopg2.connect("dbname=seek user=seek_user password=secret host=localhost")

# Open a cursor to perform database operations
cur = conn.cursor()

# Create a table
cur.execute("""
    CREATE TABLE documents(
        id SERIAL PRIMARY KEY,
        timestamp TIMESTAMP,
        speaker_name TEXT,
        document_number TEXT,
        document_title TEXT,
        transcribed_text TEXT,
        vector_id TEXT
    )
""")

# Commit changes and close
conn.commit()
cur.close()
conn.close()


DuplicateTable: relation "documents" already exists


# Milvus Setup

In [3]:
fmt = "\n=== {:30} ===\n"
search_latency_fmt = "search latency = {:.4f}s"
num_entities, dim = 3000, 768

## Connect to milvus and check for collection

In [4]:
print(fmt.format("start connecting to Milvus"))
connections.connect("default", host="localhost", port="19530")

collection_name = "seek_milvus"
has = utility.has_collection(collection_name)
print(f"Does collection hello_milvus exist in Milvus: {has}")


=== start connecting to Milvus     ===

Does collection hello_milvus exist in Milvus: True


## Drop a collection

In [5]:
utility.drop_collection(collection_name=collection_name)

## Create a collection

In [6]:
fields = [
    FieldSchema(name="id", dtype=DataType.INT64, is_primary=True, auto_id=False),
    FieldSchema(name="embeddings", dtype=DataType.FLOAT_VECTOR, dim=dim)
]

schema = CollectionSchema(fields, "SEEK collection")

print(fmt.format(f"Create collection {collection_name}"))
seek_milvus = Collection("seek_milvus", schema, consistency_level="Strong")
print(seek_milvus)



=== Create collection seek_milvus  ===

<Collection>:
-------------
<name>: seek_milvus
<description>: SEEK collection
<schema>: {'auto_id': False, 'description': 'SEEK collection', 'fields': [{'name': 'id', 'description': '', 'type': <DataType.INT64: 5>, 'is_primary': True, 'auto_id': False}, {'name': 'embeddings', 'description': '', 'type': <DataType.FLOAT_VECTOR: 101>, 'params': {'dim': 768}}]}



# Insert sample data into databases

## Init bert model and tokenizer

In [7]:
model = BertModel.from_pretrained('bert-base-uncased')
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

In [8]:
# Function to generate a random vector
def generate_random_vector(dim):
    return np.random.rand(dim).tolist()

# Function to generate a random sentence
def generate_random_sentence(length):
    # List of words to choose from
    words = ['Lorem', 'ipsum', 'dolor', 'sit', 'amet', 'consectetur', 'adipiscing', 'elit']
    
    # Choose `length` random words from the list
    sentence = ' '.join(random.choices(words, k=length))
    
    return sentence

def generate_embedding(transcribed_text):
    inputs = tokenizer(transcribed_text, return_tensors='pt', truncation=True, padding=True)
    with torch.no_grad():
        outputs = model(**inputs)
    embedding = outputs.last_hidden_state[:,0,:].numpy()
    return embedding

# Function to generate dummy metadata
def generate_dummy_metadata(doc_id):
    return {
        'timestamp': datetime.now(),  # current time
        'speaker_name': f'Speaker {doc_id}',
        'document_number': f'{doc_id}',
        'document_title': f'Document {doc_id}',
        'transcribed_text': generate_random_sentence(10),  # 10-word random sentence
    }

# Generate sample data
num_documents = 10
dimension = 768  # adjust based on your actual vector size
documents = []
vectors = []

for i in range(num_documents):
    documents.append(generate_dummy_metadata(i+1))
    # vectors.append(generate_embedding(documents[i]['transcribed_text']))

# Insert documents into postgres

In [9]:
# Connect to PostgreSQL
conn = psycopg2.connect("dbname=seek user=seek_user password=secret host=localhost")
cur = conn.cursor()

for i in range(num_documents):
    document = documents[i]
    
    cur.execute(
        """INSERT INTO documents (timestamp, speaker_name, document_number, document_title, transcribed_text) VALUES (%s, %s, %s, %s, %s)""", 
        (document["timestamp"], document['speaker_name'], document['document_number'], document['document_title'], document['transcribed_text'])
    )

# Commit changes and close
conn.commit()
cur.close()
conn.close()

# Insert embeddings into milvus and add their primary keys to postgres

In [10]:
# Connect to PostgreSQL
conn = psycopg2.connect("dbname=seek user=seek_user password=secret host=localhost")
cur = conn.cursor()

cur.execute("SELECT * FROM documents")
rows = cur.fetchall()

for row in rows:
    # print(row)
    vector = generate_embedding(row[4])
    vectors.append(vector)
    # vector = [[i for i in range(dim)]]
    # print(vector)
    # print(vector.shape)
    
    # Insert vector into Milvus
    insert_result = seek_milvus.insert([row[:1], vector])
    # print(insert_result)
    vector_id = insert_result.primary_keys
    # print(row[0])
    # print(vector_id[0])
    
    # Insert the metadata into PostgreSQL, associating it with the vector ID
    cur.execute(
    """UPDATE documents SET vector_id = %s WHERE id = %s;""",
    (vector_id[0], row[0])
    )

# Commit changes and close
conn.commit()
cur.close()
conn.close()


In [11]:
index = {
    "index_type": "IVF_FLAT",
    "metric_type": "L2",
    "params": {"nlist": 4},
}
seek_milvus.create_index("embeddings", index)

Status(code=0, message=)

In [12]:
seek_milvus.load()

vectors_to_search = vectors
search_params = {
    "metric_type": "L2",
}

In [19]:
req = seek_milvus.query(
  expr = "id in [1,2,3,4,5]", 
  output_fields = ["id", "embeddings"],
  consistency_level="Strong"
)
print(req[0]["embeddings"])

[-0.46242395, -0.050364517, -0.045184888, -0.17691249, 0.19596969, -0.15355249, -0.023614809, 0.105557375, -0.16633056, -0.09887867, -0.27729324, 0.030476727, -0.22062927, 0.11748879, 0.0066870414, 0.29390785, -0.37094992, 0.21022706, 0.26591018, -0.19143388, -0.0918508, 0.0039157853, -0.36896157, -0.04024879, -0.17029385, 0.14413275, 0.07915094, -0.009215295, -0.10106864, 0.3066476, 0.012469265, -0.35490695, 0.14067282, 0.5461102, 0.06541644, -0.06686671, -0.3384702, -0.2572647, -0.25846598, -0.19252339, 0.30429158, 0.4651209, 0.099395886, -0.095372535, -0.0065876334, -0.16021244, -1.9181131, -0.12223138, -0.18995999, 0.12041888, -0.07511855, 0.24182287, 0.025086306, 0.047215648, 0.15513998, 0.0632875, -0.39360902, 0.36621088, -0.07443306, 0.004952247, 0.18635808, -0.16992176, -0.18787841, -0.0029490415, 0.23319012, -0.04139031, -0.082231216, 0.21151543, -0.16877705, 0.4081261, -0.15038732, 0.007652172, -0.014964982, 0.19862598, 0.08431607, -0.14502966, 0.079121634, 0.13058771, -0.328

In [21]:
start_time = time()
result = seek_milvus.search([req[0]["embeddings"]], "embeddings", search_params, limit=1)
end_time = time()

In [48]:
print(result)
print(result[0].ids[0])


["['id: 11, distance: 0.0, entity: {}']"]
11


In [47]:
# Connect to your postgres DB
conn = psycopg2.connect("dbname=seek user=seek_user password=secret host=localhost")

# Open a cursor to perform database operations
cur = conn.cursor()

# Create a table
cur.execute(
    """SELECT * FROM documents WHERE vector_id = %s""",
    (str(result[0].ids[0]),))

# Fetch and print the result
rows = cur.fetchall()
for row in rows:
    print(row)

# Commit changes and close
conn.commit()
cur.close()
conn.close()

(11, datetime.datetime(2024, 2, 24, 12, 59, 17, 245731), 'Speaker 1', '1', 'Document 1', 'consectetur Lorem ipsum amet Lorem adipiscing amet sit dolor elit', '11')


In [ ]:
cur.execute(
    """UPDATE documents SET vector_id = %s WHERE id = %s;""",
    (vector_id[0], row[0])
    )

seek_milvus.load()

vectors_to_search = vectors
search_params = {
    "metric_type": "L2",
}

req = seek_milvus.query(
  expr = "id in [1,2,3,4,5]", 
  output_fields = ["id", "embeddings"],
  consistency_level="Strong"
)
print(req[0]["embeddings"])

start_time = time()
result = seek_milvus.search([req[0]["embeddings"]], "embeddings", search_params, limit=1)
end_time = time()

print(result)
print(result[0].ids[0])

# Connect to your postgres DB
conn = psycopg2.connect("dbname=seek user=seek_user password=secret host=localhost")

# Open a cursor to perform database operations
cur = conn.cursor()

# Create a table
cur.execute(
    """SELECT * FROM documents WHERE vector_id = %s""",
    (str(result[0].ids[0]),))

# Fetch and print the result
rows = cur.fetchall()
for row in rows:
    print(row)

# Commit changes and close
conn.commit()
cur.close()
conn.close()